# Install and Import Dependencies


In [1]:
!pip install tenseal syft pennylane
!pip install protobuf==3.20.3

  Using cached protobuf-5.29.6-cp310-abi3-win_amd64.whl.metadata (592 bytes)
Using cached protobuf-5.29.6-cp310-abi3-win_amd64.whl (435 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.3
    Uninstalling protobuf-3.20.3:
      Successfully uninstalled protobuf-3.20.3
  Using cached protobuf-3.20.3-cp310-cp310-win_amd64.whl.metadata (698 bytes)
Using cached protobuf-3.20.3-cp310-cp310-win_amd64.whl (904 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googleapis-common-protos 1.75.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
opentelemetry-proto 1.29.0 requires protobuf<6.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.


In [2]:
import os
import math
import copy
import random
import pickle
import time
from collections import OrderedDict, defaultdict
from typing import List, Tuple, Dict, Optional, Callable, Union, cast

os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sn

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, Subset

import torchvision
from torchvision import datasets, transforms

import pennylane as qml
import tenseal as ts
import syft as sy

from io import BytesIO
from functools import reduce
from logging import WARNING
from sklearn.metrics import roc_curve, auc, confusion_matrix


# Utility Functions


In [3]:
def choice_device(device):
    if torch.cuda.is_available() and device != "cpu":
        device = "cuda:0"
    elif (
        torch.backends.mps.is_available()
        and torch.backends.mps.is_built()
        and device != "cpu"
    ):
        device = "mps"
    else:
        device = "cpu"
    return device


def classes_string(name_dataset):
    if name_dataset == "cifar":
        return (
            "plane",
            "car",
            "bird",
            "cat",
            "deer",
            "dog",
            "frog",
            "horse",
            "ship",
            "truck",
        )
    elif name_dataset == "svhn":
        return tuple(str(i) for i in range(10))  # SVHN has 10 classes (digits 0-9)
    elif name_dataset == "caltech101":
        return tuple([f"class_{i}" for i in range(101)])  # Caltech101 has 101 classes
    elif name_dataset == "stanfordcars":
        return tuple([f"class_{i}" for i in range(196)])  # StanfordCars has 196 classes
    elif name_dataset == "fashion_mnist":
        return (
            "T-shirt/top",
            "Trouser",
            "Pullover",
            "Dress",
            "Coat",
            "Sandal",
            "Shirt",
            "Sneaker",
            "Bag",
            "Ankle boot",
        )
    else:
        raise ValueError(f"Unsupported dataset: {name_dataset}")


def save_matrix(y_true, y_pred, path, classes):
    y_true_mapped = [classes[label] for label in y_true]
    y_pred_mapped = [classes[label] for label in y_pred]
    cf_matrix_normalized = confusion_matrix(
        y_true_mapped, y_pred_mapped, labels=classes, normalize="all"
    )
    cf_matrix_round = np.round(cf_matrix_normalized, 2)
    df_cm = pd.DataFrame(
        cf_matrix_round, index=[i for i in classes], columns=[i for i in classes]
    )
    plt.figure(figsize=(12, 7))
    sn.heatmap(df_cm, annot=True)
    plt.xlabel("Predicted label", fontsize=13)
    plt.ylabel("True label", fontsize=13)
    plt.title("Confusion Matrix", fontsize=15)
    plt.savefig(path)
    plt.close()


def save_roc(targets, y_proba, path, nbr_classes):
    y_true = np.zeros(shape=(len(targets), nbr_classes))
    for i in range(len(targets)):
        y_true[i, targets[i]] = 1
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    for i in range(nbr_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true[:, i], y_proba[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    fpr["micro"], tpr["micro"], _ = roc_curve(y_true.ravel(), y_proba.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(nbr_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(nbr_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= nbr_classes
    fpr["macro"] = all_fpr
    tpr["macro"] = mean_tpr
    roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])
    plt.figure()
    plt.plot(
        fpr["micro"],
        tpr["micro"],
        label=f"micro-average ROC curve (area = {roc_auc['micro']:.2f})",
        color="deeppink",
        linestyle=":",
        linewidth=4,
    )
    plt.plot(
        fpr["macro"],
        tpr["macro"],
        label=f"macro-average ROC curve (area = {roc_auc['macro']:.2f})",
        color="navy",
        linestyle=":",
        linewidth=4,
    )
    lw = 2
    for i in range(nbr_classes):
        plt.plot(
            fpr[i],
            tpr[i],
            lw=lw,
            label=f"ROC curve of class {i} (area = {roc_auc[i]:.2f})",
        )
    plt.plot([0, 1], [0, 1], "k--", lw=lw, label="Worst case")
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("Receiver operating characteristic (ROC) Curve OvR")
    plt.legend(loc="lower right")
    plt.savefig(path)
    plt.close()


def save_graphs(path_save, local_epoch, results, end_file=""):
    os.makedirs(path_save, exist_ok=True)
    print("Saving graphs in ", path_save)
    plot_graph(
        [[*range(local_epoch)]] * 2,
        [results["train_acc"], results["val_acc"]],
        "Epochs",
        "Accuracy (%)",
        ["Training accuracy", "Validation accuracy"],
        "Accuracy curves",
        path_save + "Accuracy_curves" + end_file,
    )
    plot_graph(
        [[*range(local_epoch)]] * 2,
        [results["train_loss"], results["val_loss"]],
        "Epochs",
        "Loss",
        ["Training loss", "Validation loss"],
        "Loss curves",
        path_save + "Loss_curves" + end_file,
    )


def plot_graph(
    list_xplot, list_yplot, x_label, y_label, curve_labels, title, path=None
):
    lw = 2
    plt.figure()
    for i in range(len(curve_labels)):
        plt.plot(list_xplot[i], list_yplot[i], lw=lw, label=curve_labels[i])
    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.title(title)
    if curve_labels:
        plt.legend(loc="lower right")
    if path:
        plt.savefig(path)
    plt.close()


def get_parameters2(net, context_client=None) -> List[np.ndarray]:
    if context_client:
        encrypted_tensor = crypte(net.state_dict(), context_client)
        return [layer.get_weight() for layer in encrypted_tensor]
    return [val.cpu().numpy() for _, val in net.state_dict().items()]


def set_parameters(net, parameters: List[np.ndarray], context_client=None):
    state_dict = net.state_dict()
    params_dict = zip(state_dict.keys(), parameters)
    if context_client:
        secret_key = context_client.secret_key()
        dico = {k: deserialized_layer(k, v, context_client) for k, v in params_dict}
        new_state_dict = OrderedDict()
        for k, v in dico.items():
            if isinstance(v, CryptedLayer):
                decrypted = v.decrypt(secret_key)
                shape = state_dict[k].shape
                new_state_dict[k] = torch.Tensor(np.array(decrypted).reshape(shape))
            else:
                new_state_dict[k] = torch.Tensor(v.get_weight())
    else:
        new_state_dict = OrderedDict({k: torch.Tensor(v) for k, v in params_dict})
    net.load_state_dict(new_state_dict, strict=True)
    print("Updated model parameters")

# Security-related classes and functions


In [4]:
class Layer:
    def __init__(self, name_layer, weight):
        self.name = name_layer
        self.weight_array = weight

    def get_name(self):
        return self.name

    def get_weight(self):
        return self.weight_array

    def __add__(self, other):
        weights = other.get_weight() if isinstance(other, Layer) else other
        return Layer(self.name, self.weight_array + weights)

    def __sub__(self, other):
        weights = other.get_weight() if isinstance(other, Layer) else other
        return Layer(self.name, self.weight_array - weights)

    def __mul__(self, other):
        weights = other.get_weight() if isinstance(other, Layer) else other
        return Layer(self.name, self.weight_array * weights)

    def __truediv__(self, other):
        weights = other.get_weight() if isinstance(other, Layer) else other
        weights = self.weight_array * (1 / weights)
        return Layer(self.name, weights)

    def __len__(self):
        somme = 1
        for elem in self.weight_array.shape:
            somme *= elem
        return somme

    def shape(self):
        return self.weight_array.shape

    def sum(self, axis=0):
        return Layer(f"sum_{self.name}", self.weight_array.sum(axis=axis))

    def mean(self, axis=0):
        weights = self.weight_array.sum(axis=axis) * (1 / self.weight_array.shape[axis])
        return Layer(f"sum_{self.name}", weights)

    def decrypt(self, sk=None):
        return self.weight_array.tolist()

    def serialize(self):
        return {self.name: self.weight_array}


class CryptedLayer(Layer):
    def __init__(self, name_layer, weight, contexte=None):
        super(CryptedLayer, self).__init__(name_layer, weight)
        if isinstance(weight, (ts.tensors.CKKSTensor, bytes)):
            self.weight_array = weight
        else:
            self.weight_array = ts.ckks_tensor(contexte, weight.cpu().detach().numpy())

    def __add__(self, other):
        weights = other.get_weight() if isinstance(other, CryptedLayer) else other
        return CryptedLayer(self.name, self.weight_array + weights)

    def __sub__(self, other):
        weights = other.get_weight() if isinstance(other, CryptedLayer) else other
        return CryptedLayer(self.name, self.weight_array - weights)

    def __mul__(self, other):
        weights = other.get_weight() if isinstance(other, CryptedLayer) else other
        return CryptedLayer(self.name, self.weight_array * weights)

    def __truediv__(self, other):
        try:
            weights = other.get_weight() if isinstance(other, CryptedLayer) else other
            weights = self.weight_array * (1 / weights)
        except:
            print("Error: division operator not supported by SEAL")
            weights = []
        return CryptedLayer(self.name, weights)

    def shape(self):
        return self.weight_array.shape

    def sum(self, axis=0):
        return CryptedLayer(f"sum_{self.name}", self.weight_array.sum(axis=axis))

    def mean(self, axis=0):
        weights = self.weight_array.sum(axis=axis) * (1 / self.weight_array.shape[axis])
        return CryptedLayer(f"sum_{self.name}", weights)

    def decrypt(self, sk=None):
        return (
            self.weight_array.decrypt(sk).tolist()
            if sk
            else self.weight_array.decrypt().tolist()
        )

    def serialize(self):
        return {self.name: self.weight_array.serialize()}


def context():
    cont = ts.context(
        ts.SCHEME_TYPE.CKKS,
        poly_modulus_degree=8192,
        coeff_mod_bit_sizes=[60, 40, 40, 60],
    )
    cont.generate_galois_keys()
    cont.global_scale = 2**40
    return cont


def crypte(client_w, context_c):
    encrypted = []
    for name_layer, weight_array in client_w.items():
        if name_layer in {"fc4.weight", "classifier.weight"}:
            encrypted.append(CryptedLayer(name_layer, weight_array, context_c))
        else:
            encrypted.append(Layer(name_layer, weight_array))
    return encrypted


def read_query(file_path):
    if os.path.exists(file_path):
        with open(file_path, "rb") as file:
            query_str = pickle.load(file)
        contexte = query_str["contexte"]
        del query_str["contexte"]
        return query_str, contexte
    else:
        print(f"File {file_path} does not exist")
        return None, None


def write_query(file_path, client_query):
    with open(file_path, "wb") as file:
        encode_str = pickle.dumps(client_query)
        file.write(encode_str)


def deserialized_layer(name_layer, weight_array, ctx):
    if isinstance(weight_array, bytes):
        return CryptedLayer(name_layer, ts.ckks_tensor_from(ctx, weight_array), ctx)
    elif isinstance(weight_array, ts.tensors.CKKSTensor):
        return CryptedLayer(name_layer, weight_array, ctx)
    else:
        return Layer(name_layer, weight_array)


# Data setup


In [5]:
NORMALIZE_DICT = {
    "cifar": dict(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    "svhn": dict(mean=(0.4377, 0.4438, 0.4728), std=(0.1980, 0.2010, 0.1970)),
    "caltech101": dict(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    "stanfordcars": dict(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    "fashion_mnist": dict(mean=(0.2860,), std=(0.3530,)),
}


def _dataset_targets(dataset_obj) -> np.ndarray:
    """Return integer labels for torchvision-style datasets."""
    if hasattr(dataset_obj, "targets"):
        labels = dataset_obj.targets
    elif hasattr(dataset_obj, "labels"):
        labels = dataset_obj.labels
    else:
        raise AttributeError("Dataset does not expose .targets or .labels")
    if torch.is_tensor(labels):
        labels = labels.cpu().numpy()
    return np.asarray(labels, dtype=np.int64)


def iid_partition_indices(
    n_samples: int,
    num_clients: int,
    seed: int,
) -> List[np.ndarray]:
    rng = np.random.default_rng(seed)
    indices = rng.permutation(n_samples)
    return [np.asarray(x, dtype=np.int64) for x in np.array_split(indices, num_clients)]


def dirichlet_partition_indices(
    labels: np.ndarray,
    num_clients: int,
    alpha: float,
    seed: int,
    min_client_samples: int = 100,
    max_retries: int = 200,
) -> List[np.ndarray]:
    """
    Label-skew partition used in the AdeptHEQ-FL paper.
    A small alpha (0.1) creates strongly non-IID client label distributions.
    """
    if alpha <= 0:
        raise ValueError("Dirichlet alpha must be > 0")

    rng = np.random.default_rng(seed)
    classes = np.unique(labels)
    average_size = len(labels) / num_clients

    for _ in range(max_retries):
        client_indices: List[List[int]] = [[] for _ in range(num_clients)]

        for cls in classes:
            cls_indices = np.where(labels == cls)[0].copy()
            rng.shuffle(cls_indices)

            proportions = rng.dirichlet(np.full(num_clients, alpha))

            # Mild balancing prevents a few clients from absorbing almost all samples.
            balance_mask = np.asarray(
                [len(idx) < average_size for idx in client_indices],
                dtype=np.float64,
            )
            proportions = proportions * balance_mask
            if proportions.sum() == 0:
                proportions = np.ones(num_clients, dtype=np.float64)
            proportions = proportions / proportions.sum()

            split_points = (
                np.cumsum(proportions)[:-1] * len(cls_indices)
            ).astype(int)
            class_splits = np.split(cls_indices, split_points)

            for cid, split in enumerate(class_splits):
                client_indices[cid].extend(split.tolist())

        sizes = [len(idx) for idx in client_indices]
        if min(sizes) >= min_client_samples:
            output = []
            for idx in client_indices:
                arr = np.asarray(idx, dtype=np.int64)
                rng.shuffle(arr)
                output.append(arr)
            return output

    raise RuntimeError(
        f"Could not create a Dirichlet partition with at least "
        f"{min_client_samples} samples/client after {max_retries} retries. "
        "Reduce min_client_samples or increase alpha."
    )


def _split_client_train_val(
    indices: np.ndarray,
    val_fraction: float,
    seed: int,
) -> Tuple[np.ndarray, np.ndarray]:
    if not 0 < val_fraction < 1:
        raise ValueError("val_fraction must be between 0 and 1")
    rng = np.random.default_rng(seed)
    shuffled = np.asarray(indices, dtype=np.int64).copy()
    rng.shuffle(shuffled)

    n_val = max(1, int(round(len(shuffled) * val_fraction)))
    n_val = min(n_val, len(shuffled) - 1)
    return shuffled[n_val:], shuffled[:n_val]


def _make_dataset(
    dataset: str,
    root: str,
    train: bool,
    transformer,
):
    if dataset == "cifar":
        return datasets.CIFAR10(root + dataset, train=train, download=True, transform=transformer)
    if dataset == "svhn":
        split = "train" if train else "test"
        return datasets.SVHN(root + "svhn", split=split, download=True, transform=transformer)
    if dataset == "fashion_mnist":
        return datasets.FashionMNIST(
            root + "fashion_mnist",
            train=train,
            download=True,
            transform=transformer,
        )
    if dataset == "stanfordcars":
        split = "train" if train else "test"
        return datasets.StanfordCars(
            root + "stanfordcars",
            split=split,
            download=True,
            transform=transformer,
        )
    if dataset == "caltech101":
        split = "train" if train else "test"
        return datasets.ImageFolder(root + f"caltech101/{split}", transform=transformer)
    raise ValueError(f"Unsupported dataset: {dataset}")


def load_datasets(
    num_clients: int,
    batch_size: int,
    resize: Optional[int],
    seed: int,
    num_workers: int,
    splitter: float = 10.0,
    dataset: str = "fashion_mnist",
    data_path: str = "./data/",
    partition_mode: str = "dirichlet",
    dirichlet_alpha: float = 0.1,
    min_client_samples: int = 100,
):
    """
    Returns:
        trainloaders, valloaders, testloader, partition_stats

    splitter is the local validation percentage, kept for compatibility
    with the original notebook.
    """
    list_transforms = [
        transforms.ToTensor(),
        transforms.Normalize(**NORMALIZE_DICT[dataset]),
    ]

    if dataset in ["caltech101", "stanfordcars"] and resize is not None:
        list_transforms = [transforms.Resize((resize, resize))] + list_transforms
    elif dataset == "svhn":
        list_transforms = [transforms.Resize((32, 32))] + list_transforms

    transformer = transforms.Compose(list_transforms)
    trainset = _make_dataset(dataset, data_path, train=True, transformer=transformer)
    testset = _make_dataset(dataset, data_path, train=False, transformer=transformer)

    labels = _dataset_targets(trainset)
    if partition_mode == "dirichlet":
        client_indices = dirichlet_partition_indices(
            labels=labels,
            num_clients=num_clients,
            alpha=dirichlet_alpha,
            seed=seed,
            min_client_samples=min_client_samples,
        )
    elif partition_mode == "iid":
        client_indices = iid_partition_indices(len(trainset), num_clients, seed)
    else:
        raise ValueError("partition_mode must be 'dirichlet' or 'iid'")

    trainloaders, valloaders = [], []
    partition_stats = []

    for cid, indices in enumerate(client_indices):
        train_idx, val_idx = _split_client_train_val(
            indices,
            val_fraction=splitter / 100.0,
            seed=seed + 10_000 + cid,
        )

        generator = torch.Generator().manual_seed(seed + cid)
        trainloaders.append(
            DataLoader(
                Subset(trainset, train_idx.tolist()),
                batch_size=batch_size,
                shuffle=True,
                num_workers=num_workers,
                pin_memory=torch.cuda.is_available(),
                generator=generator,
            )
        )
        valloaders.append(
            DataLoader(
                Subset(trainset, val_idx.tolist()),
                batch_size=batch_size,
                shuffle=False,
                num_workers=num_workers,
                pin_memory=torch.cuda.is_available(),
            )
        )

        class_counts = np.bincount(labels[indices], minlength=len(np.unique(labels)))
        partition_stats.append(
            {
                "client": cid,
                "total": int(len(indices)),
                "train": int(len(train_idx)),
                "val": int(len(val_idx)),
                "class_counts": class_counts.tolist(),
            }
        )

    testloader = DataLoader(
        testset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    print(f"Partition mode: {partition_mode}")
    if partition_mode == "dirichlet":
        print(f"Dirichlet alpha: {dirichlet_alpha}")
    for stat in partition_stats:
        print(
            f"Client {stat['client']:02d} | total={stat['total']:5d} | "
            f"class_counts={stat['class_counts']}"
        )

    return trainloaders, valloaders, testloader, partition_stats

# ============================================================
# Balanced rotating per-round client data
# ============================================================

class BalancedRotatingClientScheduler:
    """
    Creates a class-balanced local dataset for every participating client
    in every FL round.

    Guarantees in reuse_scope="per_client":
      1. Every client receives exactly samples_per_class_per_client
         samples from every class in each round.
      2. Clients do not share a training block within the same round.
      3. The same client does not receive the same training block again
         until all available class blocks have been cycled through.
      4. A block may be reused by a different client in a later round.

    In reuse_scope="global":
      No training block is reused by any client across rounds, but the
      maximum number of rounds is much smaller.
    """

    def __init__(
        self,
        trainset,
        num_clients: int,
        batch_size: int,
        samples_per_class_per_client: int = 100,
        val_per_class_per_client: int = 20,
        seed: int = 42,
        num_workers: int = 0,
        reuse_scope: str = "per_client",
    ):
        self.trainset = trainset
        self.num_clients = int(num_clients)
        self.batch_size = int(batch_size)
        self.samples_per_class_per_client = int(
            samples_per_class_per_client
        )
        self.val_per_class_per_client = int(
            val_per_class_per_client
        )
        self.seed = int(seed)
        self.num_workers = int(num_workers)
        self.reuse_scope = reuse_scope.lower()

        if self.samples_per_class_per_client <= 0:
            raise ValueError(
                "samples_per_class_per_client must be positive"
            )
        if self.val_per_class_per_client < 0:
            raise ValueError(
                "val_per_class_per_client cannot be negative"
            )
        if self.reuse_scope not in {"per_client", "global"}:
            raise ValueError(
                "reuse_scope must be 'per_client' or 'global'"
            )

        labels = _dataset_targets(trainset)
        self.labels = labels
        self.classes = np.unique(labels).astype(int).tolist()
        self.num_classes = len(self.classes)

        rng = np.random.default_rng(self.seed)
        self.class_blocks = {}
        self.validation_indices = {
            cid: []
            for cid in range(self.num_clients)
        }

        block_counts = []

        for cls in self.classes:
            cls_indices = np.where(labels == cls)[0].astype(
                np.int64
            )
            rng.shuffle(cls_indices)

            required_val = (
                self.num_clients
                * self.val_per_class_per_client
            )
            if required_val >= len(cls_indices):
                raise ValueError(
                    f"Class {cls} has only {len(cls_indices)} samples, "
                    f"but {required_val} were requested for validation."
                )

            validation_pool = cls_indices[:required_val]
            training_pool = cls_indices[required_val:]

            for cid in range(self.num_clients):
                start = cid * self.val_per_class_per_client
                stop = start + self.val_per_class_per_client
                self.validation_indices[cid].extend(
                    validation_pool[start:stop].tolist()
                )

            n_blocks = (
                len(training_pool)
                // self.samples_per_class_per_client
            )
            if n_blocks < self.num_clients:
                raise ValueError(
                    f"Not enough class-{cls} blocks for "
                    f"{self.num_clients} clients."
                )

            usable = (
                n_blocks
                * self.samples_per_class_per_client
            )
            training_pool = training_pool[:usable]

            self.class_blocks[cls] = [
                training_pool[
                    block_id
                    * self.samples_per_class_per_client:
                    (block_id + 1)
                    * self.samples_per_class_per_client
                ].copy()
                for block_id in range(n_blocks)
            ]
            block_counts.append(n_blocks)

        self.num_blocks = min(block_counts)

        # Choose a stride that is coprime to num_blocks. This ensures
        # that a fixed client cycles through all blocks before repeating.
        stride = self.num_clients + 1
        while math.gcd(stride, self.num_blocks) != 1:
            stride += 1
        self.block_stride = stride

        self.max_global_nonrepeat_rounds = (
            self.num_blocks // self.num_clients
        )
        self.max_per_client_nonrepeat_rounds = (
            self.num_blocks
        )

        self.valloaders = []
        for cid in range(self.num_clients):
            val_indices = np.asarray(
                self.validation_indices[cid],
                dtype=np.int64,
            )
            val_rng = np.random.default_rng(
                self.seed + 50_000 + cid
            )
            val_rng.shuffle(val_indices)

            self.valloaders.append(
                DataLoader(
                    Subset(
                        self.trainset,
                        val_indices.tolist(),
                    ),
                    batch_size=self.batch_size,
                    shuffle=False,
                    num_workers=self.num_workers,
                    pin_memory=torch.cuda.is_available(),
                )
            )

    def _block_id(
        self,
        round_index: int,
        cid: int,
    ) -> int:
        if self.reuse_scope == "global":
            block_id = (
                round_index * self.num_clients + cid
            )
            if block_id >= self.num_blocks:
                raise RuntimeError(
                    "Strict global non-reuse has exhausted the "
                    "available class blocks. With the current settings, "
                    f"the maximum is {self.max_global_nonrepeat_rounds} "
                    "rounds. Reduce samples per class per client, reduce "
                    "the number of clients, or use reuse_scope='per_client'."
                )
            return block_id

        # Per-client non-repeat schedule. Same-round clients use
        # distinct blocks. A client does not repeat until num_blocks rounds.
        return (
            cid + round_index * self.block_stride
        ) % self.num_blocks

    def get_round_loaders(
        self,
        round_index: int,
        selected_clients: List[int],
    ):
        loaders = {}
        stats = {}

        for cid in selected_clients:
            block_id = self._block_id(
                round_index=round_index,
                cid=cid,
            )

            indices = []
            for cls in self.classes:
                indices.extend(
                    self.class_blocks[cls][block_id].tolist()
                )

            indices = np.asarray(
                indices,
                dtype=np.int64,
            )
            round_rng = np.random.default_rng(
                self.seed
                + 100_000
                + 10_000 * round_index
                + cid
            )
            round_rng.shuffle(indices)

            generator = torch.Generator().manual_seed(
                self.seed
                + 200_000
                + 10_000 * round_index
                + cid
            )

            loaders[cid] = DataLoader(
                Subset(
                    self.trainset,
                    indices.tolist(),
                ),
                batch_size=self.batch_size,
                shuffle=True,
                num_workers=self.num_workers,
                pin_memory=torch.cuda.is_available(),
                generator=generator,
            )

            class_counts = np.bincount(
                self.labels[indices],
                minlength=self.num_classes,
            )
            stats[cid] = {
                "client": int(cid),
                "round": int(round_index + 1),
                "block_id": int(block_id),
                "total": int(len(indices)),
                "class_counts": class_counts.tolist(),
            }

        return loaders, stats


def load_balanced_rotating_datasets(
    num_clients: int,
    batch_size: int,
    seed: int,
    num_workers: int,
    dataset: str = "fashion_mnist",
    data_path: str = "./data/",
    samples_per_class_per_client: int = 100,
    val_per_class_per_client: int = 20,
    reuse_scope: str = "per_client",
):
    """
    Builds the rotating balanced scheduler, fixed balanced local
    validation loaders, and the standard global test loader.
    """
    transformer = transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Normalize(
                **NORMALIZE_DICT[dataset]
            ),
        ]
    )

    trainset = _make_dataset(
        dataset,
        data_path,
        train=True,
        transformer=transformer,
    )
    testset = _make_dataset(
        dataset,
        data_path,
        train=False,
        transformer=transformer,
    )

    scheduler = BalancedRotatingClientScheduler(
        trainset=trainset,
        num_clients=num_clients,
        batch_size=batch_size,
        samples_per_class_per_client=(
            samples_per_class_per_client
        ),
        val_per_class_per_client=(
            val_per_class_per_client
        ),
        seed=seed,
        num_workers=num_workers,
        reuse_scope=reuse_scope,
    )

    testloader = DataLoader(
        testset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    print("Partition mode: balanced_rotating")
    print(
        "Training samples/client/round: "
        f"{samples_per_class_per_client} per class "
        f"({samples_per_class_per_client * len(scheduler.classes)} total)"
    )
    print(
        "Validation samples/client: "
        f"{val_per_class_per_client} per class "
        f"({val_per_class_per_client * len(scheduler.classes)} total)"
    )
    print(f"Reuse scope: {reuse_scope}")
    print(f"Class blocks available: {scheduler.num_blocks}")
    print(f"Rotation stride: {scheduler.block_stride}")
    print(
        "Maximum no-repeat rounds for each client: "
        f"{scheduler.max_per_client_nonrepeat_rounds}"
    )
    print(
        "Maximum strict globally non-repeating rounds: "
        f"{scheduler.max_global_nonrepeat_rounds}"
    )

    validation_stats = []
    for cid, loader in enumerate(scheduler.valloaders):
        indices = np.asarray(
            loader.dataset.indices,
            dtype=np.int64,
        )
        counts = np.bincount(
            scheduler.labels[indices],
            minlength=len(scheduler.classes),
        )
        validation_stats.append(
            {
                "client": cid,
                "total": int(len(indices)),
                "class_counts": counts.tolist(),
            }
        )

    return (
        scheduler,
        scheduler.valloaders,
        testloader,
        validation_stats,
    )


# Training and testing functions


In [6]:
def test(
    model: torch.nn.Module,
    dataloader: torch.utils.data.DataLoader,
    loss_fn: Union[torch.nn.Module, Tuple],
    device: torch.device,
):
    model.eval()
    test_loss, test_acc = 0, 0
    y_pred = []
    y_true = []
    y_proba = []
    softmax = nn.Softmax(dim=1)
    with torch.inference_mode():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            output = model(images)
            probas_output = softmax(output)
            y_proba.extend(probas_output.detach().cpu().numpy())
            loss = loss_fn(output, labels)
            test_loss += loss.item()
            labels = labels.data.cpu().numpy()
            y_true.extend(labels)
            preds = np.argmax(output.detach().cpu().numpy(), axis=1)
            y_pred.extend(preds)
            acc = (preds == labels).mean()
            test_acc += acc
    y_proba = np.array(y_proba)
    test_loss = test_loss / len(dataloader)
    test_acc = test_acc / len(dataloader)
    return test_loss, test_acc * 100, y_pred, y_true, y_proba


def train_step(
    model: torch.nn.Module,
    dataloader: torch.utils.data.DataLoader,
    loss_fn: Union[torch.nn.Module, Tuple],
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> Tuple[float, float]:
    model.train()
    train_loss, train_acc = 0, 0
    for batch, (images, labels) in enumerate(dataloader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        output = model(images)
        loss = loss_fn(output, labels)
        train_loss += loss.item()
        loss.backward()
        optimizer.step()
        y_pred_class = torch.argmax(torch.softmax(output, dim=1), dim=1)
        train_acc += (y_pred_class == labels).sum().item() / len(output)
    train_loss = train_loss / len(dataloader)
    train_acc = train_acc / len(dataloader)
    return train_loss, train_acc * 100


def train(
    model: torch.nn.Module,
    train_dataloader: torch.utils.data.DataLoader,
    test_dataloader: torch.utils.data.DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: Union[torch.nn.Module, Tuple],
    epochs: int,
    device: torch.device,
) -> Dict[str, List]:
    results = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    for epoch in range(epochs):
        train_loss, train_acc = train_step(
            model, train_dataloader, loss_fn, optimizer, device
        )
        val_loss, val_acc, *_ = test(model, test_dataloader, loss_fn, device)
        print(
            f"\tTrain Epoch: {epoch + 1} \tTrain_loss: {train_loss:.4f} | Train_acc: {train_acc:.4f} % | "
            f"Validation_loss: {val_loss:.4f} | Validation_acc: {val_acc:.4f} %"
        )
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["val_loss"].append(val_loss)
        results["val_acc"].append(val_acc)
    return results


def serialize_ndarray(ndarray):
    if isinstance(ndarray, ts.tensors.CKKSTensor):
        return ndarray.serialize()
    elif isinstance(ndarray, torch.Tensor):
        return serialize_ndarray(ndarray.cpu().detach().numpy())
    else:
        bytes_io = BytesIO()
        np.save(bytes_io, ndarray, allow_pickle=False)
        return bytes_io.getvalue()


def deserialize_ndarray(tensor, context):
    try:
        return ts.ckks_tensor_from(context, tensor)
    except:
        bytes_io = BytesIO(tensor)
        return np.load(bytes_io, allow_pickle=False)


def serialize_parameters(parameters):
    return [serialize_ndarray(param) for param in parameters]


def deserialize_parameters(serialized_params, context):
    return [deserialize_ndarray(param, context) for param in serialized_params]


def privatize_accuracy(true_acc: float, N: int, ε=1.0):
    sensitivity = 1.0 / N
    noise = np.random.laplace(0, sensitivity / ε)
    return np.clip(true_acc + noise, 0, 1)


def accuracy_weights(accuracies: List[float], τ=0.5) -> List[float]:
    scaled_acc = [a / τ for a in accuracies]
    max_scaled = max(scaled_acc)
    exp_acc = [np.exp(a - max_scaled) for a in scaled_acc]
    sum_exp = sum(exp_acc)
    return [e / sum_exp for e in exp_acc]


def compute_difference_norm(new_param, prev_param, context):
    if isinstance(new_param, ts.tensors.CKKSTensor):
        new_dec = new_param.decrypt(context.secret_key()).tolist()
        prev_dec = prev_param.decrypt(context.secret_key()).tolist()
        diff = np.array(new_dec) - np.array(prev_dec)
    else:
        if isinstance(new_param, torch.Tensor):
            new_param = new_param.cpu().numpy()
        if isinstance(prev_param, torch.Tensor):
            prev_param = prev_param.cpu().numpy()
        diff = new_param - prev_param
    return np.linalg.norm(diff)


def aggregate_serialized(results, context, τ=0.5):
    accuracies = [dp_acc for _, dp_acc in results]
    weights = accuracy_weights(accuracies, τ)

    weights_results = [
        (deserialize_parameters(serialized_params, context), w)
        for (serialized_params, _), w in zip(results, weights)
    ]

    aggregated_params = []
    for layer_idx in range(len(weights_results[0][0])):
        layer_updates = [weights[layer_idx] for weights, _ in weights_results]
        if isinstance(layer_updates[0], ts.tensors.CKKSTensor):
            weighted_sum = sum([layer * w for layer, w in zip(layer_updates, weights)])
        else:
            weighted_sum = sum([layer * w for layer, w in zip(layer_updates, weights)])
        aggregated_params.append(weighted_sum)
    return serialize_parameters(aggregated_params)

# Main experiment setup


In [7]:
# ============================================================
# Experiment configuration
# ============================================================

he = False                 # Keep OFF for architecture/optimizer ablations.
use_dp = False             # Keep OFF for architecture/optimizer ablations.
data_path = "data/"
dataset = "fashion_mnist"
seed = 42
num_workers = 0
batch_size = 32
splitter = 10.0            # Local validation percentage.
device = "gpu"
number_clients = 10
save_results = "results/FL/"
model_save = "fashionmnist_BalancedRotating_AdeptHEQ.pt"

# Use a quick smoke test first. Set False for the full 20-round study.
SMOKE_TEST = False
max_epochs = 1 if SMOKE_TEST else 3
rounds = 2 if SMOKE_TEST else 12

# Balanced-data diagnostic: use every client in each round.
frac_fit = 1.0
min_fit_clients = number_clients

# Server aggregation mode for diagnosis:
# "uniform", "sample_size", "quality", or "hybrid".
AGGREGATION_MODE = "uniform"

# AdeptHEQ aggregation / privacy settings.
TAU_AGG = 0.5
DP_EPSILON = 1.0
USE_QUALITY_EMA = False
QUALITY_EMA_RHO = 0.8

# Data allocation. "balanced_rotating" is the professor-requested test.
PARTITION_MODE = "balanced_rotating"  # "balanced_rotating", "dirichlet", or "iid"
DIRICHLET_ALPHA = 0.1
MIN_CLIENT_SAMPLES = 100
BALANCED_SAMPLES_PER_CLASS_PER_CLIENT = 100
BALANCED_VAL_PER_CLASS_PER_CLIENT = 20
# "per_client": each client never repeats its own block; blocks may move to
# another client in a later round. "global": no block is reused anywhere.
BALANCED_REUSE_SCOPE = "per_client"

# Architecture and local optimizer ablations.
MODEL_VARIANT = "parallel_qbank"     # "original" or "parallel_qbank"
QUANTUM_TRAINING = "naive"           # Start with "naive"; test "lazy" second.
USE_FEDPROX = False
FEDPROX_MU = 1e-3

# Block-wise server EMA.
USE_SERVER_EMA = False
SERVER_EMA_BETA_CLASSICAL = 0.9
SERVER_EMA_BETA_QUANTUM = 0.0        # 0.0 means use the fresh aggregated quantum block.

# Operational layer sparing.
USE_LAYER_SPARING = False
IMPORTANCE_EMA_ALPHA = 0.9
FREEZE_THRESHOLD = 0.001
FREEZE_WARMUP_ROUNDS = 3
FREEZE_PATIENCE = 2

# Local optimization.
LR_CLASSICAL = 1e-3
LR_QUANTUM = 5e-2

# New parallel data-reuploading QBank.
LATENT_DIM = 16
N_QUBITS = 4
K_CIRCUITS = 4
R_UNIQUE = 2
RU_LAYERS = 2

# Lazy quantum refresh.
TAU_DRIFT = 0.15
REFRESH_EVERY = 25
BETA_QGRAD = 0.5
GAMMA_STALE = 0.9

# CKKS is supported by the integrated loop but should be enabled only
# after architecture and optimizer ablations are complete.
ENCRYPTED_FINAL_WEIGHT = "classifier.weight"
path_public_key = "server_key.pkl"
secret_path = "secret.pkl"

DEVICE = torch.device(choice_device(device))
CLASSES = classes_string(dataset)


# ============================================================
# Original AdeptHEQ-FL local model retained as a baseline
# ============================================================

_original_dev = qml.device("default.qubit", wires=N_QUBITS)
_original_weight_shapes = {"weights": (2, N_QUBITS, 3)}


@qml.qnode(_original_dev, interface="torch", diff_method="backprop")
def original_quantum_net(inputs, weights):
    qml.AmplitudeEmbedding(
        features=inputs,
        wires=range(N_QUBITS),
        pad_with=0.0,
        normalize=True,
    )
    qml.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]


class OriginalAdeptNet(nn.Module):
    """Original 4-qubit, 2-layer Fashion-MNIST AdeptHEQ model."""

    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Flatten(),
            nn.Linear(256 * 3 * 3, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 2**N_QUBITS),
        )
        self.qnn = qml.qnn.TorchLayer(original_quantum_net, _original_weight_shapes)
        self.fc4 = nn.Linear(N_QUBITS, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.network(x)
        x = self.qnn(x)
        return self.fc4(x)


# ============================================================
# New parallel data-reuploading QBank
# ============================================================

_reupload_dev = qml.device("default.qubit", wires=N_QUBITS)


@qml.qnode(_reupload_dev, interface="torch", diff_method="backprop")
def data_reuploading_circuit(inputs, weights):
    """
    inputs:  [..., RU_LAYERS * N_QUBITS]
    weights: [RU_LAYERS, N_QUBITS]
    """
    for layer_idx in range(RU_LAYERS):
        start = layer_idx * N_QUBITS
        stop = (layer_idx + 1) * N_QUBITS

        qml.AngleEmbedding(
            inputs[..., start:stop],
            wires=range(N_QUBITS),
            rotation="Y",
        )

        for qubit_idx in range(N_QUBITS):
            qml.RY(weights[layer_idx, qubit_idx], wires=qubit_idx)

        # Ring entanglement: 0->1->2->3->0.
        for qubit_idx in range(N_QUBITS - 1):
            qml.CNOT(wires=[qubit_idx, qubit_idx + 1])
        qml.CNOT(wires=[N_QUBITS - 1, 0])

    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]


class AdeptCNNEncoder(nn.Module):
    """
    Keeps the original AdeptHEQ convolutional backbone.
    Only the final projection is changed to a compact latent vector.
    """

    def __init__(self, latent_dim: int = LATENT_DIM):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Flatten(),
            nn.Linear(256 * 3 * 3, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, latent_dim),
            nn.Tanh(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


class ParallelDataReuploadingQBank(nn.Module):
    """
    Four parallel circuits with cyclic sharing across two unique
    quantum parameter tensors. Adapters and q_weights are treated
    as one quantum-coupled parameter group.
    """

    def __init__(
        self,
        latent_dim: int = LATENT_DIM,
        k_circuits: int = K_CIRCUITS,
        r_unique: int = R_UNIQUE,
    ):
        super().__init__()
        if r_unique > k_circuits:
            raise ValueError("R_UNIQUE cannot exceed K_CIRCUITS")

        self.k_circuits = k_circuits
        self.r_unique = r_unique
        features_per_circuit = RU_LAYERS * N_QUBITS

        self.adapters = nn.ModuleList(
            [
                nn.Linear(latent_dim, features_per_circuit)
                for _ in range(k_circuits)
            ]
        )

        self.q_weights = nn.ParameterList(
            [
                nn.Parameter(
                    0.01 * torch.randn(RU_LAYERS, N_QUBITS)
                )
                for _ in range(r_unique)
            ]
        )

    def _shared_weight_id(self, circuit_id: int) -> int:
        return circuit_id % self.r_unique

    @staticmethod
    def _stack_qnode_output(output) -> torch.Tensor:
        if isinstance(output, (tuple, list)):
            return torch.stack(list(output), dim=-1)
        return output

    def _run_one_circuit(
        self,
        features: torch.Tensor,
        weights: torch.Tensor,
        detach_circuit: bool,
    ) -> torch.Tensor:
        original_device = features.device

        if detach_circuit:
            with torch.no_grad():
                output = data_reuploading_circuit(
                    features.detach().to("cpu"),
                    weights.detach().to("cpu"),
                )
        else:
            # .to("cpu") remains differentiable, so fresh gradients flow
            # back to GPU parameters if the classical model is on CUDA.
            output = data_reuploading_circuit(
                features.to("cpu"),
                weights.to("cpu"),
            )

        output = self._stack_qnode_output(output)
        return output.to(device=original_device, 
                         dtype=features.dtype,
                         )

    def forward(
        self,
        latent: torch.Tensor,
        detach_circuit: bool = False,
    ) -> torch.Tensor:
        quantum_features = []

        for circuit_id, adapter in enumerate(self.adapters):
            angles = torch.tanh(adapter(latent)) * math.pi
            shared_weights = self.q_weights[self._shared_weight_id(circuit_id)]
            q_out = self._run_one_circuit(
                angles,
                shared_weights,
                detach_circuit=detach_circuit,
            )
            quantum_features.append(q_out)

        return torch.cat(quantum_features, dim=1)


class ParallelQBankAdeptNet(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.encoder = AdeptCNNEncoder(latent_dim=LATENT_DIM)
        self.qbank = ParallelDataReuploadingQBank(latent_dim=LATENT_DIM)

        fused_dim = LATENT_DIM + K_CIRCUITS * N_QUBITS
        self.fusion = nn.Sequential(
            nn.Linear(fused_dim, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(0.1),
        )
        self.classifier = nn.Linear(64, num_classes)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.encoder(x)

    def quantum_features(
        self,
        latent: torch.Tensor,
        detach_circuit: bool = False,
    ) -> torch.Tensor:
        return self.qbank(latent, detach_circuit=detach_circuit)

    def classify(
        self,
        latent: torch.Tensor,
        quantum_features: torch.Tensor,
    ) -> torch.Tensor:
        # Defensive normalization at the quantum–classical boundary.
        quantum_features = quantum_features.to(
        device=latent.device,
        dtype=latent.dtype,
        )
        fused = torch.cat([latent, quantum_features], dim=1)
        return self.classifier(self.fusion(fused))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        latent = self.encode(x)
        q_features = self.quantum_features(latent, detach_circuit=False)
        return self.classify(latent, q_features)


def build_model(num_classes: int):
    if MODEL_VARIANT == "original":
        if QUANTUM_TRAINING == "lazy":
            raise ValueError(
                "Lazy refresh is implemented for the parallel QBank model. "
                "Use QUANTUM_TRAINING='naive' with MODEL_VARIANT='original'."
            )
        return OriginalAdeptNet(num_classes=num_classes)

    if MODEL_VARIANT == "parallel_qbank":
        return ParallelQBankAdeptNet(num_classes=num_classes)

    raise ValueError("MODEL_VARIANT must be 'original' or 'parallel_qbank'")


In [20]:
# ============================================================
# Quantum–classical interface sanity check
# ============================================================

_dtype_test_model = build_model(
    num_classes=len(CLASSES)
).to(DEVICE)

_dtype_test_batch, _ = next(iter(trainloaders[0]))
_dtype_test_batch = _dtype_test_batch[:2].to(DEVICE)

_dtype_test_model.eval()

with torch.no_grad():
    if isinstance(_dtype_test_model, ParallelQBankAdeptNet):
        _latent = _dtype_test_model.encode(_dtype_test_batch)
        _q_features = _dtype_test_model.quantum_features(
            _latent,
            detach_circuit=True,
        )
        _logits = _dtype_test_model.classify(
            _latent,
            _q_features,
        )

        print("Input dtype:            ", _dtype_test_batch.dtype)
        print("Latent dtype/device:    ", _latent.dtype, _latent.device)
        print("Quantum dtype/device:   ", _q_features.dtype, _q_features.device)
        print("Fusion weight dtype:    ", _dtype_test_model.fusion[0].weight.dtype)
        print("Classifier output:      ", _logits.shape, _logits.dtype)

        assert _latent.dtype == torch.float32
        assert _q_features.dtype == _latent.dtype
        assert _q_features.device == _latent.device
        assert _logits.dtype == _dtype_test_model.classifier.weight.dtype
        assert _logits.shape == (2, len(CLASSES))

        print("✓ Quantum–classical dtype interface passed.")

del _dtype_test_model

Input dtype:             torch.float32
Latent dtype/device:     torch.float32 cuda:0
Quantum dtype/device:    torch.float32 cuda:0
Fusion weight dtype:     torch.float32
Classifier output:       torch.Size([2, 10]) torch.float32
✓ Quantum–classical dtype interface passed.


# Integrated LazyQ, FedProx, EMA, and Operational Layer-Sparing Helpers


In [8]:
# ============================================================
# Integrated local-training, communication, aggregation,
# EMA, and layer-sparing helpers
# ============================================================

def clone_state_dict(
    state_dict: Dict[str, torch.Tensor],
) -> OrderedDict:
    return OrderedDict(
        (name, tensor.detach().cpu().clone())
        for name, tensor in state_dict.items()
    )


def is_quantum_coupled_name(name: str) -> bool:
    """
    New model: adapters + PQC weights.
    Original baseline: TorchLayer qnn parameters.
    """
    return (
        name.startswith("qbank.adapters.")
        or name.startswith("qbank.q_weights.")
        or name.startswith("qnn.")
    )


def parameter_block_name(name: str) -> str:
    """Group a module's weight and bias under one freezing decision."""
    return name.rsplit(".", 1)[0]


def set_trainability_from_mask(
    model: nn.Module,
    active_mask: Dict[str, bool],
) -> None:
    for name, parameter in model.named_parameters():
        # Quantum-coupled parameters are always active in version 1.
        parameter.requires_grad = (
            True if is_quantum_coupled_name(name)
            else active_mask.get(name, True)
        )


def split_parameter_groups(model: nn.Module):
    classical_named = []
    quantum_named = []

    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if is_quantum_coupled_name(name):
            quantum_named.append((name, parameter))
        else:
            classical_named.append((name, parameter))

    return classical_named, quantum_named


def _build_optimizer(
    named_parameters,
    optimizer_name: str,
    learning_rate: float,
):
    parameters = [parameter for _, parameter in named_parameters]
    if not parameters:
        return None

    optimizer_name = optimizer_name.lower()
    if optimizer_name == "adam":
        return torch.optim.Adam(parameters, lr=learning_rate)
    if optimizer_name == "sgd":
        return torch.optim.SGD(parameters, lr=learning_rate)
    raise ValueError("optimizer_name must be 'adam' or 'sgd'")


def exact_evaluate(
    model: nn.Module,
    dataloader: DataLoader,
    loss_fn: nn.Module,
    device: torch.device,
):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    y_true, y_pred, y_proba = [], [], []
    softmax = nn.Softmax(dim=1)

    with torch.inference_mode():
        for images, labels in dataloader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(images)
            batch_size_now = labels.size(0)

            total_loss += loss_fn(logits, labels).item() * batch_size_now
            predictions = logits.argmax(dim=1)

            total_correct += (predictions == labels).sum().item()
            total_examples += batch_size_now

            y_true.extend(labels.cpu().tolist())
            y_pred.extend(predictions.cpu().tolist())
            y_proba.extend(softmax(logits).cpu().numpy())

    return (
        total_loss / max(total_examples, 1),
        100.0 * total_correct / max(total_examples, 1),
        y_pred,
        y_true,
        np.asarray(y_proba),
    )


def _reference_on_device(
    reference_state: Dict[str, torch.Tensor],
    model: nn.Module,
) -> Dict[str, torch.Tensor]:
    return {
        name: reference_state[name].to(
            parameter.device,
            dtype=parameter.dtype,
        )
        for name, parameter in model.named_parameters()
    }


def _proximal_penalty(
    named_parameters,
    reference: Dict[str, torch.Tensor],
    mu: float,
) -> torch.Tensor:
    if not named_parameters or mu <= 0:
        if named_parameters:
            return named_parameters[0][1].new_zeros(())
        return torch.tensor(0.0, device=DEVICE)

    penalty = named_parameters[0][1].new_zeros(())
    for name, parameter in named_parameters:
        penalty = penalty + torch.sum(
            (parameter - reference[name]) ** 2
        )
    return 0.5 * mu * penalty


def _flat_parameters(named_parameters) -> torch.Tensor:
    if not named_parameters:
        return torch.empty(0, device=DEVICE)
    return torch.cat(
        [parameter.detach().reshape(-1) for _, parameter in named_parameters]
    )


def train_local_standard(
    model: nn.Module,
    trainloader: DataLoader,
    reference_state: Dict[str, torch.Tensor],
    local_epochs: int,
    device: torch.device,
):
    """
    Standard local training used for the original AdeptHEQ baseline.
    FedProx can be enabled without changing the aggregation protocol.
    """
    model.train()
    criterion = nn.CrossEntropyLoss()

    active_named = [
        (name, parameter)
        for name, parameter in model.named_parameters()
        if parameter.requires_grad
    ]
    optimizer = _build_optimizer(
        active_named,
        optimizer_name="adam",
        learning_rate=LR_CLASSICAL,
    )
    reference = _reference_on_device(reference_state, model)

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for _ in range(local_epochs):
        for images, labels in trainloader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            task_loss = criterion(logits, labels)

            if USE_FEDPROX:
                prox = _proximal_penalty(
                    active_named,
                    reference,
                    FEDPROX_MU,
                )
            else:
                prox = task_loss.new_zeros(())

            loss = task_loss + prox
            loss.backward()
            optimizer.step()

            total_loss += task_loss.item() * labels.size(0)
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_examples += labels.size(0)

    metrics = {
        "train_loss": total_loss / max(total_examples, 1),
        "train_acc": 100.0 * total_correct / max(total_examples, 1),
        "quantum_forward_passes": 0,
        "fresh_quantum_backward_passes": 0,
        "lazy_quantum_steps": 0,
        "refresh_ratio": 1.0,
        "max_quantum_drift": 0.0,
    }
    return metrics


def train_local_parallel_qbank(
    model: ParallelQBankAdeptNet,
    trainloader: DataLoader,
    reference_state: Dict[str, torch.Tensor],
    local_epochs: int,
    device: torch.device,
):
    """
    Lazy quantum state is created inside this function and therefore
    resets every federated round, as required.
    """
    model.train()
    criterion = nn.CrossEntropyLoss()
    reference = _reference_on_device(reference_state, model)

    classical_named, quantum_named = split_parameter_groups(model)
    classical_optimizer = _build_optimizer(
        classical_named,
        optimizer_name="adam",
        learning_rate=LR_CLASSICAL,
    )
    quantum_optimizer = _build_optimizer(
        quantum_named,
        optimizer_name="sgd",
        learning_rate=LR_QUANTUM,
    )

    if quantum_optimizer is None:
        raise RuntimeError("The parallel QBank has no active quantum-coupled parameters")

    anchor = _flat_parameters(quantum_named).clone()
    ema_task_gradients = None
    last_refresh_step = -REFRESH_EVERY
    local_step = 0

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    quantum_forward_passes = 0
    fresh_quantum_backward_passes = 0
    lazy_quantum_steps = 0
    max_quantum_drift = 0.0

    for _ in range(local_epochs):
        for images, labels in trainloader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            current_q = _flat_parameters(quantum_named)
            drift = torch.linalg.vector_norm(current_q - anchor).item()
            max_quantum_drift = max(max_quantum_drift, drift)

            if QUANTUM_TRAINING == "naive":
                do_refresh = True
            elif QUANTUM_TRAINING == "lazy":
                threshold_hit = drift > TAU_DRIFT
                forced_refresh = (
                    local_step == 0
                    or (local_step - last_refresh_step) >= REFRESH_EVERY
                    or ema_task_gradients is None
                )
                do_refresh = threshold_hit or forced_refresh
            else:
                raise ValueError("QUANTUM_TRAINING must be 'naive' or 'lazy'")

            if do_refresh:
                if classical_optimizer is not None:
                    classical_optimizer.zero_grad(set_to_none=True)
                quantum_optimizer.zero_grad(set_to_none=True)

                latent = model.encode(images)
                q_features = model.quantum_features(
                    latent,
                    detach_circuit=False,
                )
                quantum_forward_passes += K_CIRCUITS

                logits = model.classify(latent, q_features)
                task_loss = criterion(logits, labels)

                if USE_FEDPROX:
                    all_active_named = classical_named + quantum_named
                    prox = _proximal_penalty(
                        all_active_named,
                        reference,
                        FEDPROX_MU,
                    )
                else:
                    prox = task_loss.new_zeros(())

                total_objective = task_loss + prox
                total_objective.backward()

                # Extract the fresh task gradient while removing the
                # analytically known FedProx contribution. This lets the
                # lazy steps reuse only the task gradient and add a fresh
                # proximal correction at every step.
                fresh_task_gradients = []
                for name, parameter in quantum_named:
                    if parameter.grad is None:
                        total_gradient = torch.zeros_like(parameter)
                    else:
                        total_gradient = parameter.grad.detach().clone()

                    if USE_FEDPROX:
                        prox_gradient = FEDPROX_MU * (
                            parameter.detach() - reference[name]
                        )
                    else:
                        prox_gradient = torch.zeros_like(parameter)

                    fresh_task_gradients.append(
                        total_gradient - prox_gradient
                    )

                if ema_task_gradients is None:
                    ema_task_gradients = [
                        gradient.clone()
                        for gradient in fresh_task_gradients
                    ]
                else:
                    ema_task_gradients = [
                        BETA_QGRAD * old_gradient
                        + (1.0 - BETA_QGRAD) * fresh_gradient
                        for old_gradient, fresh_gradient in zip(
                            ema_task_gradients,
                            fresh_task_gradients,
                        )
                    ]

                if classical_optimizer is not None:
                    classical_optimizer.step()
                quantum_optimizer.step()

                anchor = _flat_parameters(quantum_named).clone()
                last_refresh_step = local_step
                fresh_quantum_backward_passes += 1

            else:
                # The direct classical skip path still trains the encoder
                # and fusion head. Adapters and PQC weights use cached
                # quantum-coupled gradients.
                if classical_optimizer is not None:
                    classical_optimizer.zero_grad(set_to_none=True)

                latent = model.encode(images)
                q_features = model.quantum_features(
                    latent,
                    detach_circuit=True,
                )
                quantum_forward_passes += K_CIRCUITS

                logits = model.classify(latent, q_features)
                task_loss = criterion(logits, labels)

                if USE_FEDPROX and classical_named:
                    prox_classical = _proximal_penalty(
                        classical_named,
                        reference,
                        FEDPROX_MU,
                    )
                else:
                    prox_classical = task_loss.new_zeros(())

                classical_objective = task_loss + prox_classical
                if (
                    classical_optimizer is not None
                    and classical_objective.requires_grad
                ):
                    classical_objective.backward()
                    classical_optimizer.step()

                quantum_optimizer.zero_grad(set_to_none=True)
                age = max(local_step - last_refresh_step, 0)
                stale_scale = GAMMA_STALE ** age

                for (name, parameter), cached_task_gradient in zip(
                    quantum_named,
                    ema_task_gradients,
                ):
                    gradient = stale_scale * cached_task_gradient

                    if USE_FEDPROX:
                        gradient = gradient + FEDPROX_MU * (
                            parameter.detach() - reference[name]
                        )

                    parameter.grad = gradient.detach().clone()

                quantum_optimizer.step()
                lazy_quantum_steps += 1

            total_loss += task_loss.item() * labels.size(0)
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_examples += labels.size(0)
            local_step += 1

    metrics = {
        "train_loss": total_loss / max(total_examples, 1),
        "train_acc": 100.0 * total_correct / max(total_examples, 1),
        "quantum_forward_passes": quantum_forward_passes,
        "fresh_quantum_backward_passes": fresh_quantum_backward_passes,
        "lazy_quantum_steps": lazy_quantum_steps,
        "refresh_ratio": (
            fresh_quantum_backward_passes / max(local_step, 1)
        ),
        "max_quantum_drift": max_quantum_drift,
    }
    return metrics


def train_local_model(
    model: nn.Module,
    trainloader: DataLoader,
    reference_state: Dict[str, torch.Tensor],
    local_epochs: int,
    device: torch.device,
):
    if isinstance(model, ParallelQBankAdeptNet):
        return train_local_parallel_qbank(
            model,
            trainloader,
            reference_state,
            local_epochs,
            device,
        )

    return train_local_standard(
        model,
        trainloader,
        reference_state,
        local_epochs,
        device,
    )


def initialize_he_contexts():
    """
    Same CKKS parameters as AdeptHEQ-FL.
    Returns a secret context for simulation-side decryption and a
    public context for client encryption/server homomorphic addition.
    """
    secret_context = context()
    public_context = ts.context_from(
        secret_context.serialize(save_secret_key=False)
    )
    return secret_context, public_context


def build_client_payload(
    model: nn.Module,
    active_mask: Dict[str, bool],
    use_he: bool,
    public_context=None,
):
    """
    Only active tensors are transmitted.
    The final classifier weight alone is encrypted when HE is enabled.
    """
    payload = {}
    total_bytes = 0

    for name, tensor in model.state_dict().items():
        is_active = (
            True if is_quantum_coupled_name(name)
            else active_mask.get(name, True)
        )
        if not is_active:
            continue

        array = tensor.detach().cpu().numpy()

        if use_he and name == ENCRYPTED_FINAL_WEIGHT:
            if public_context is None:
                raise ValueError("public_context is required when HE is enabled")
            encrypted = ts.ckks_tensor(public_context, array)
            serialized = encrypted.serialize()
            payload[name] = {
                "encrypted": True,
                "data": serialized,
                "shape": tuple(array.shape),
                "dtype": str(array.dtype),
            }
            total_bytes += len(serialized)
        else:
            payload[name] = {
                "encrypted": False,
                "data": array,
                "shape": tuple(array.shape),
                "dtype": str(array.dtype),
            }
            total_bytes += array.nbytes

    return payload, total_bytes


def aggregate_client_payloads(
    client_payloads,
    aggregation_weights: List[float],
    previous_raw_state: Dict[str, torch.Tensor],
    use_he: bool,
    public_context=None,
    secret_context=None,
):
    """
    AdeptHEQ accuracy-weighted aggregation, now using sparse
    named payloads so frozen tensors do not need to be transmitted.
    """
    if len(client_payloads) != len(aggregation_weights):
        raise ValueError("Payload count and weight count do not match")

    aggregated_state = clone_state_dict(previous_raw_state)
    transmitted_names = sorted(
        set().union(*(payload.keys() for payload in client_payloads))
    )

    for name in transmitted_names:
        entries = [
            payload[name]
            for payload in client_payloads
            if name in payload
        ]
        entry_weights = [
            weight
            for payload, weight in zip(
                client_payloads,
                aggregation_weights,
            )
            if name in payload
        ]

        weight_sum = sum(entry_weights)
        entry_weights = [weight / weight_sum for weight in entry_weights]

        if entries[0]["encrypted"]:
            if not use_he or public_context is None or secret_context is None:
                raise ValueError("HE contexts are required for encrypted aggregation")

            encrypted_tensors = [
                ts.ckks_tensor_from(public_context, entry["data"])
                for entry in entries
            ]

            encrypted_sum = encrypted_tensors[0] * entry_weights[0]
            for encrypted_tensor, weight in zip(
                encrypted_tensors[1:],
                entry_weights[1:],
            ):
                encrypted_sum = encrypted_sum + encrypted_tensor * weight

            decrypted = np.asarray(
                encrypted_sum.decrypt(secret_context.secret_key()),
                dtype=np.float32,
            ).reshape(entries[0]["shape"])

            aggregated_state[name] = torch.as_tensor(
                decrypted,
                dtype=previous_raw_state[name].dtype,
            )
        else:
            weighted_array = np.zeros(
                entries[0]["shape"],
                dtype=np.float64,
            )
            for entry, weight in zip(entries, entry_weights):
                weighted_array += entry["data"].astype(np.float64) * weight

            aggregated_state[name] = torch.as_tensor(
                weighted_array,
                dtype=previous_raw_state[name].dtype,
            )

    return aggregated_state


def apply_blockwise_server_ema(
    raw_state: Dict[str, torch.Tensor],
    previous_ema_state: Dict[str, torch.Tensor],
):
    ema_state = OrderedDict()

    for name, raw_tensor in raw_state.items():
        beta = (
            SERVER_EMA_BETA_QUANTUM
            if is_quantum_coupled_name(name)
            else SERVER_EMA_BETA_CLASSICAL
        )

        if not USE_SERVER_EMA or beta <= 0:
            ema_state[name] = raw_tensor.detach().cpu().clone()
        else:
            ema_state[name] = (
                beta * previous_ema_state[name]
                + (1.0 - beta) * raw_tensor
            ).detach().cpu()

    return ema_state


def initialize_layer_sparing_state(model: nn.Module):
    active_mask = {
        name: True
        for name, _ in model.named_parameters()
    }

    blocks = {
        parameter_block_name(name)
        for name, _ in model.named_parameters()
        if not is_quantum_coupled_name(name)
    }

    importance_history = {
        block: None
        for block in blocks
    }
    low_importance_counts = {
        block: 0
        for block in blocks
    }
    frozen_blocks = set()

    return (
        active_mask,
        importance_history,
        low_importance_counts,
        frozen_blocks,
    )


def update_layer_sparing_mask(
    model: nn.Module,
    raw_state: Dict[str, torch.Tensor],
    previous_raw_state: Dict[str, torch.Tensor],
    active_mask: Dict[str, bool],
    importance_history: Dict[str, float],
    low_importance_counts: Dict[str, int],
    frozen_blocks: set,
    completed_round: int,
):
    if not USE_LAYER_SPARING:
        return (
            active_mask.copy(),
            importance_history,
            low_importance_counts,
            frozen_blocks,
            {},
        )

    block_differences = defaultdict(list)

    for name, _ in model.named_parameters():
        if is_quantum_coupled_name(name):
            continue

        block = parameter_block_name(name)
        difference = (
            raw_state[name] - previous_raw_state[name]
        ).reshape(-1)
        block_differences[block].append(difference)

    current_norms = {}
    for block, differences in block_differences.items():
        flat_difference = torch.cat(differences)
        current_norms[block] = torch.linalg.vector_norm(
            flat_difference
        ).item()

        previous_importance = importance_history.get(block)
        if previous_importance is None:
            importance_history[block] = current_norms[block]
        else:
            importance_history[block] = (
                IMPORTANCE_EMA_ALPHA * previous_importance
                + (1.0 - IMPORTANCE_EMA_ALPHA) * current_norms[block]
            )

        if completed_round >= FREEZE_WARMUP_ROUNDS:
            if importance_history[block] < FREEZE_THRESHOLD:
                low_importance_counts[block] = (
                    low_importance_counts.get(block, 0) + 1
                )
            else:
                low_importance_counts[block] = 0

            if low_importance_counts[block] >= FREEZE_PATIENCE:
                frozen_blocks.add(block)

    next_active_mask = {}
    for name, _ in model.named_parameters():
        if is_quantum_coupled_name(name):
            next_active_mask[name] = True
        else:
            next_active_mask[name] = (
                parameter_block_name(name) not in frozen_blocks
            )

    return (
        next_active_mask,
        importance_history,
        low_importance_counts,
        frozen_blocks,
        current_norms,
    )


def full_model_parameter_bytes(model: nn.Module) -> int:
    return sum(
        parameter.numel() * parameter.element_size()
        for parameter in model.parameters()
    )


def select_round_clients(
    rng: np.random.Generator,
    num_clients: int,
    fraction: float,
    minimum: int,
) -> List[int]:
    selected_count = max(
        minimum,
        int(math.ceil(fraction * num_clients)),
    )
    selected_count = min(selected_count, num_clients)

    selected = rng.choice(
        num_clients,
        size=selected_count,
        replace=False,
    )
    return sorted(int(cid) for cid in selected.tolist())


# Main Experiment


In [9]:
# ============================================================
# Integrated LazyQ-AdeptHEQ federated experiment
# ============================================================

# Reproducibility
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

print("=" * 88)
print("Integrated LazyQ-AdeptHEQ configuration")
print(f"Device:                 {DEVICE}")
print(f"Model variant:          {MODEL_VARIANT}")
print(f"Quantum training:       {QUANTUM_TRAINING}")
print(f"Partition mode:         {PARTITION_MODE}")
if PARTITION_MODE == "dirichlet":
    print(f"Dirichlet alpha:        {DIRICHLET_ALPHA}")
elif PARTITION_MODE == "balanced_rotating":
    print(
        f"Balanced samples:       "
        f"{BALANCED_SAMPLES_PER_CLASS_PER_CLIENT}/class/client/round"
    )
    print(f"Balanced reuse scope:   {BALANCED_REUSE_SCOPE}")
print(f"Client fraction/round:  {frac_fit}")
print(f"Aggregation mode:       {AGGREGATION_MODE}")
print(f"FedProx:                {USE_FEDPROX} (mu={FEDPROX_MU})")
print(f"Server EMA:             {USE_SERVER_EMA}")
print(f"Quality EMA:            {USE_QUALITY_EMA}")
print(f"Layer sparing:          {USE_LAYER_SPARING}")
print(f"Differential privacy:   {use_dp}")
print(f"CKKS encryption:        {he}")
print(f"Smoke test:             {SMOKE_TEST}")
print("=" * 88)

# Create HE contexts only when the secure experiment is enabled.
if he:
    secret_context, public_context = initialize_he_contexts()
else:
    secret_context, public_context = None, None

# Client data.
if PARTITION_MODE == "balanced_rotating":
    (
        balanced_scheduler,
        valloaders,
        testloader,
        partition_stats,
    ) = load_balanced_rotating_datasets(
        num_clients=number_clients,
        batch_size=batch_size,
        seed=seed,
        num_workers=num_workers,
        dataset=dataset,
        data_path=data_path,
        samples_per_class_per_client=(
            BALANCED_SAMPLES_PER_CLASS_PER_CLIENT
        ),
        val_per_class_per_client=(
            BALANCED_VAL_PER_CLASS_PER_CLIENT
        ),
        reuse_scope=BALANCED_REUSE_SCOPE,
    )
    trainloaders = None
else:
    balanced_scheduler = None
    trainloaders, valloaders, testloader, partition_stats = load_datasets(
        num_clients=number_clients,
        batch_size=batch_size,
        resize=None,
        seed=seed,
        num_workers=num_workers,
        splitter=splitter,
        dataset=dataset,
        data_path=data_path,
        partition_mode=PARTITION_MODE,
        dirichlet_alpha=DIRICHLET_ALPHA,
        min_client_samples=MIN_CLIENT_SAMPLES,
    )

# Global/server state.
global_model = build_model(num_classes=len(CLASSES)).to(DEVICE)
criterion = nn.CrossEntropyLoss()

initial_state = clone_state_dict(global_model.state_dict())
previous_raw_state = clone_state_dict(initial_state)
server_ema_state = clone_state_dict(initial_state)
broadcast_state = clone_state_dict(initial_state)

(
    active_mask,
    importance_history,
    low_importance_counts,
    frozen_blocks,
) = initialize_layer_sparing_state(global_model)

# Per-client historical performance is maintained on the server.
quality_ema = {
    cid: None
    for cid in range(number_clients)
}

client_rng = np.random.default_rng(seed + 2026)
full_bytes_per_client = full_model_parameter_bytes(global_model)
round_history = []

print(
    f"Global model parameters: "
    f"{sum(p.numel() for p in global_model.parameters()):,}"
)
print(f"Plain full-model payload/client: {full_bytes_per_client / 1e6:.3f} MB")


def client_train_one_round(
    cid: int,
    round_trainloader: DataLoader,
    current_broadcast_state: Dict[str, torch.Tensor],
    current_active_mask: Dict[str, bool],
):
    """
    A new local model is created each FL round.
    The lazy quantum gradient cache is initialized inside
    train_local_parallel_qbank, so no stale cache crosses rounds.
    """
    local_model = build_model(num_classes=len(CLASSES)).to(DEVICE)
    local_model.load_state_dict(current_broadcast_state, strict=True)
    set_trainability_from_mask(local_model, current_active_mask)

    local_metrics = train_local_model(
        model=local_model,
        trainloader=round_trainloader,
        reference_state=current_broadcast_state,
        local_epochs=max_epochs,
        device=DEVICE,
    )

    val_loss, val_acc_percent, *_ = exact_evaluate(
        local_model,
        valloaders[cid],
        criterion,
        DEVICE,
    )
    true_val_acc = val_acc_percent / 100.0
    n_val = len(valloaders[cid].dataset)

    if use_dp:
        transmitted_quality = privatize_accuracy(
            true_val_acc,
            n_val,
            ε=DP_EPSILON,
        )
    else:
        transmitted_quality = true_val_acc

    payload, payload_bytes = build_client_payload(
        local_model,
        current_active_mask,
        use_he=he,
        public_context=public_context,
    )

    return {
        "cid": cid,
        "payload": payload,
        "payload_bytes": payload_bytes,
        "num_train_examples": len(round_trainloader.dataset),
        "true_val_acc": true_val_acc,
        "transmitted_quality": float(transmitted_quality),
        "val_loss": val_loss,
        "local_metrics": local_metrics,
    }


print(f"Training on {DEVICE}")
start_simulation = time.time()

for round_index in range(rounds):
    round_number = round_index + 1
    selected_clients = select_round_clients(
        client_rng,
        num_clients=number_clients,
        fraction=frac_fit,
        minimum=min_fit_clients,
    )

    if PARTITION_MODE == "balanced_rotating":
        round_trainloaders, round_data_stats = (
            balanced_scheduler.get_round_loaders(
                round_index=round_index,
                selected_clients=selected_clients,
            )
        )
    else:
        round_trainloaders = {
            cid: trainloaders[cid]
            for cid in selected_clients
        }
        round_data_stats = {}

    print("\n" + "#" * 88)
    print(f"Round {round_number}/{rounds}")
    print(f"Selected clients: {selected_clients}")
    print(f"Frozen blocks applied before training/upload: {sorted(frozen_blocks)}")
    print("#" * 88)

    if PARTITION_MODE == "balanced_rotating":
        for cid in selected_clients:
            stat = round_data_stats[cid]
            print(
                f"Client {cid:02d} | total={stat['total']:5d} | "
                f"block={stat['block_id']:02d} | "
                f"class_counts={stat['class_counts']}"
            )

    client_records = []
    for cid in selected_clients:
        print(f"[Client {cid}] local training started")
        record = client_train_one_round(
            cid,
            round_trainloaders[cid],
            broadcast_state,
            active_mask,
        )
        client_records.append(record)

        metrics = record["local_metrics"]
        print(
            f"[Client {cid}] train_acc={metrics['train_acc']:.2f}% | "
            f"val_acc={100.0 * record['true_val_acc']:.2f}% | "
            f"quality={record['transmitted_quality']:.4f} | "
            f"fresh_q_backwards={metrics['fresh_quantum_backward_passes']} | "
            f"lazy_steps={metrics['lazy_quantum_steps']} | "
            f"refresh_ratio={metrics['refresh_ratio']:.4f} | "
            f"payload={record['payload_bytes'] / 1e6:.3f} MB"
        )

    # EMA-smoothed client quality. With DP enabled, this is post-processing
    # of the already privatized accuracy and consumes no extra privacy budget.
    selected_quality_values = []
    for record in client_records:
        cid = record["cid"]
        new_quality = record["transmitted_quality"]

        if quality_ema[cid] is None or not USE_QUALITY_EMA:
            quality_ema[cid] = new_quality
        else:
            quality_ema[cid] = (
                QUALITY_EMA_RHO * quality_ema[cid]
                + (1.0 - QUALITY_EMA_RHO) * new_quality
            )

        selected_quality_values.append(quality_ema[cid])

    if AGGREGATION_MODE == "uniform":
        aggregation_weights = [
            1.0 / len(client_records)
            for _ in client_records
        ]
    elif AGGREGATION_MODE == "sample_size":
        sample_counts = np.asarray(
            [
                record["num_train_examples"]
                for record in client_records
            ],
            dtype=np.float64,
        )
        aggregation_weights = (
            sample_counts / sample_counts.sum()
        ).tolist()
    elif AGGREGATION_MODE == "quality":
        aggregation_weights = accuracy_weights(
            selected_quality_values,
            τ=TAU_AGG,
        )
    elif AGGREGATION_MODE == "hybrid":
        quality_component = np.asarray(
            accuracy_weights(
                selected_quality_values,
                τ=TAU_AGG,
            ),
            dtype=np.float64,
        )
        sample_counts = np.asarray(
            [
                record["num_train_examples"]
                for record in client_records
            ],
            dtype=np.float64,
        )
        sample_component = sample_counts / sample_counts.sum()
        combined = quality_component * sample_component
        aggregation_weights = (
            combined / combined.sum()
        ).tolist()
    else:
        raise ValueError(
            "AGGREGATION_MODE must be 'uniform', 'sample_size', "
            "'quality', or 'hybrid'"
        )

    print(
        f"[Round {round_number}] Smoothed qualities: "
        f"{np.round(selected_quality_values, 4).tolist()}"
    )
    print(
        f"[Round {round_number}] {AGGREGATION_MODE} weights: "
        f"{np.round(aggregation_weights, 4).tolist()}"
    )

    # Accuracy-weighted aggregation. Frozen parameters are absent from the
    # payload and are retained from previous_raw_state.
    raw_aggregated_state = aggregate_client_payloads(
        client_payloads=[
            record["payload"]
            for record in client_records
        ],
        aggregation_weights=aggregation_weights,
        previous_raw_state=previous_raw_state,
        use_he=he,
        public_context=public_context,
        secret_context=secret_context,
    )

    # Importance is computed from raw aggregates, not from EMA-smoothed
    # broadcast weights, to avoid artificially premature freezing.
    (
        next_active_mask,
        importance_history,
        low_importance_counts,
        frozen_blocks,
        block_norms,
    ) = update_layer_sparing_mask(
        model=global_model,
        raw_state=raw_aggregated_state,
        previous_raw_state=previous_raw_state,
        active_mask=active_mask,
        importance_history=importance_history,
        low_importance_counts=low_importance_counts,
        frozen_blocks=frozen_blocks,
        completed_round=round_number,
    )

    # Block-wise server EMA. Quantum-coupled parameters use beta_q=0 by
    # default so they retain the current round's adaptability.
    server_ema_state = apply_blockwise_server_ema(
        raw_state=raw_aggregated_state,
        previous_ema_state=server_ema_state,
    )
    broadcast_state = (
        clone_state_dict(server_ema_state)
        if USE_SERVER_EMA
        else clone_state_dict(raw_aggregated_state)
    )

    global_model.load_state_dict(broadcast_state, strict=True)
    (
        test_loss,
        test_acc,
        test_targets,
        test_predictions,
        _,
    ) = exact_evaluate(
        global_model,
        testloader,
        criterion,
        DEVICE,
    )
    prediction_histogram = np.bincount(
        np.asarray(test_predictions, dtype=np.int64),
        minlength=len(CLASSES),
    )

    actual_upload_bytes = sum(
        record["payload_bytes"]
        for record in client_records
    )
    full_upload_bytes = (
        full_bytes_per_client * len(selected_clients)
    )
    communication_reduction = (
        1.0 - actual_upload_bytes / max(full_upload_bytes, 1)
    )

    total_fresh_q_backwards = sum(
        record["local_metrics"]["fresh_quantum_backward_passes"]
        for record in client_records
    )
    total_lazy_steps = sum(
        record["local_metrics"]["lazy_quantum_steps"]
        for record in client_records
    )
    mean_refresh_ratio = float(
        np.mean(
            [
                record["local_metrics"]["refresh_ratio"]
                for record in client_records
            ]
        )
    )

    round_summary = {
        "round": round_number,
        "selected_clients": selected_clients,
        "test_loss": test_loss,
        "test_acc": test_acc,
        "actual_upload_bytes": actual_upload_bytes,
        "full_upload_bytes": full_upload_bytes,
        "communication_reduction": communication_reduction,
        "frozen_blocks": sorted(frozen_blocks),
        "fresh_quantum_backward_passes": total_fresh_q_backwards,
        "lazy_quantum_steps": total_lazy_steps,
        "mean_refresh_ratio": mean_refresh_ratio,
        "quality_values": selected_quality_values,
        "aggregation_weights": aggregation_weights,
        "block_norms": block_norms,
        "prediction_histogram": prediction_histogram.tolist(),
        "round_data_stats": round_data_stats,
    }
    round_history.append(round_summary)

    print(
        f"[Round {round_number}] Global test loss={test_loss:.4f} | "
        f"accuracy={test_acc:.4f}%"
    )
    print(
        f"[Round {round_number}] Test prediction histogram: "
        f"{prediction_histogram.tolist()}"
    )
    print(
        f"[Round {round_number}] Upload={actual_upload_bytes / 1e6:.3f} MB | "
        f"full-model reference={full_upload_bytes / 1e6:.3f} MB | "
        f"reduction={100.0 * communication_reduction:.2f}%"
    )
    print(
        f"[Round {round_number}] Fresh quantum backwards="
        f"{total_fresh_q_backwards} | lazy steps={total_lazy_steps} | "
        f"mean refresh ratio={mean_refresh_ratio:.4f}"
    )
    print(
        f"[Round {round_number}] Frozen blocks for next round: "
        f"{sorted(frozen_blocks)}"
    )

    previous_raw_state = clone_state_dict(raw_aggregated_state)
    active_mask = next_active_mask

simulation_time = time.time() - start_simulation
print("\n" + "=" * 88)
print(f"Federated learning completed in {simulation_time:.2f} seconds")
print(f"Final test accuracy: {round_history[-1]['test_acc']:.4f}%")
print("=" * 88)


Integrated LazyQ-AdeptHEQ configuration
Device:                 cuda:0
Model variant:          parallel_qbank
Quantum training:       naive
Partition mode:         balanced_rotating
Balanced samples:       100/class/client/round
Balanced reuse scope:   per_client
Client fraction/round:  1.0
Aggregation mode:       uniform
FedProx:                False (mu=0.001)
Server EMA:             False
Quality EMA:            False
Layer sparing:          False
Differential privacy:   False
CKKS encryption:        False
Smoke test:             False


100%|██████████| 26.4M/26.4M [03:03<00:00, 144kB/s] 
100%|██████████| 29.5k/29.5k [00:00<00:00, 154kB/s]
100%|██████████| 4.42M/4.42M [00:04<00:00, 939kB/s] 
100%|██████████| 5.15k/5.15k [00:00<00:00, 5.27MB/s]


Partition mode: balanced_rotating
Training samples/client/round: 100 per class (1000 total)
Validation samples/client: 20 per class (200 total)
Reuse scope: per_client
Class blocks available: 58
Rotation stride: 11
Maximum no-repeat rounds for each client: 58
Maximum strict globally non-repeating rounds: 5
Global model parameters: 4,022,282
Plain full-model payload/client: 16.089 MB
Training on cuda:0

########################################################################################
Round 1/12
Selected clients: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Frozen blocks applied before training/upload: []
########################################################################################
Client 00 | total= 1000 | block=00 | class_counts=[100, 100, 100, 100, 100, 100, 100, 100, 100, 100]
Client 01 | total= 1000 | block=01 | class_counts=[100, 100, 100, 100, 100, 100, 100, 100, 100, 100]
Client 02 | total= 1000 | block=02 | class_counts=[100, 100, 100, 100, 100, 100, 100, 100, 100, 100]
Clie

# Save the final model


In [14]:
if save_results:
    os.makedirs(save_results, exist_ok=True)

    checkpoint = {
        "model_state_dict": global_model.state_dict(),
        "broadcast_state": broadcast_state,
        "raw_aggregated_state": previous_raw_state,
        "round_history": round_history,
        "partition_stats": partition_stats,
        "configuration": {
            "model_variant": MODEL_VARIANT,
            "quantum_training": QUANTUM_TRAINING,
            "partition_mode": PARTITION_MODE,
            "dirichlet_alpha": DIRICHLET_ALPHA,
            "balanced_samples_per_class_per_client": (
                BALANCED_SAMPLES_PER_CLASS_PER_CLIENT
            ),
            "balanced_val_per_class_per_client": (
                BALANCED_VAL_PER_CLASS_PER_CLIENT
            ),
            "balanced_reuse_scope": BALANCED_REUSE_SCOPE,
            "aggregation_mode": AGGREGATION_MODE,
            "frac_fit": frac_fit,
            "use_fedprox": USE_FEDPROX,
            "fedprox_mu": FEDPROX_MU,
            "use_server_ema": USE_SERVER_EMA,
            "use_quality_ema": USE_QUALITY_EMA,
            "use_layer_sparing": USE_LAYER_SPARING,
            "use_dp": use_dp,
            "use_he": he,
            "rounds": rounds,
            "local_epochs": max_epochs,
            "seed": seed,
        },
    }

    torch.save(checkpoint, model_save)

    history_path = os.path.join(
        save_results,
        "integrated_round_history.pkl",
    )
    with open(history_path, "wb") as file:
        pickle.dump(round_history, file)

    print(f"Saved checkpoint: {model_save}")
    print(f"Saved round history: {history_path}")


Saved checkpoint: fashionmnist_LazyQ_AdeptHEQ.pt
Saved round history: results/FL/integrated_round_history.pkl
